In [ ]:
#Libraries installation
!pip install -U scikit-learn
!pip install librosa matplotlib umap-learn

Uploading dataset


In [ ]:
from google.colab import drive

drive.mount('/content/drive')

In [ ]:
import pandas as pd
from sklearn.preprocessing import StandardScaler

file_path = '/content/drive/My Drive/cse425/dataset_final.csv'
data = pd.read_csv(file_path)

mfcc_columns = [f'mfcc_{i}' for i in range(1, 14)]
mfcc_features = data[mfcc_columns]

tfidf_columns = [col for col in data.columns if col.startswith('tfidf_')]
tfidf_features = data[tfidf_columns]

genre_column = data['genre']

combined_features = pd.concat([mfcc_features, tfidf_features], axis=1)
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
combined_features_scaled = scaler.fit_transform(combined_features)

print(f"Shape of combined features: {combined_features_scaled.shape}")


In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.utils import to_categorical
from sklearn.cluster import AgglomerativeClustering
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.cluster import DBSCAN
from sklearn.metrics import davies_bouldin_score
from sklearn.metrics import adjusted_rand_score
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import confusion_matrix
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, davies_bouldin_score, adjusted_rand_score

Conditional VAE for Disentangled

In [ ]:
latent_dim = 2

class CVAE(models.Model):
    def __init__(self, original_dim, latent_dim, num_genres):
        super(CVAE, self).__init__()
        self.original_dim = original_dim
        self.latent_dim = latent_dim
        self.num_genres = num_genres

        self.encoder_input_concat = layers.Concatenate()
        self.encoder_dense_1 = layers.Dense(128, activation='relu')
        self.encoder_dense_2 = layers.Dense(64, activation='relu')
        self.z_mean_layer = layers.Dense(latent_dim, name='z_mean')
        self.z_log_var_layer = layers.Dense(latent_dim, name='z_log_var')

        self.decoder_dense_1 = layers.Dense(64, activation='relu')
        self.decoder_output_layer = layers.Dense(original_dim, activation='sigmoid')

    def sampling(self, args):
        z_mean, z_log_var = args
        batch = tf.shape(z_mean)[0]
        dim = tf.shape(z_mean)[1]
        epsilon = tf.keras.backend.random_normal(shape=(batch, dim))
        return z_mean + tf.exp(0.5 * z_log_var) * epsilon

    def call(self, inputs):
        features, genre_one_hot = inputs

        x = self.encoder_input_concat([features, genre_one_hot])

        x = self.encoder_dense_1(x)
        x = self.encoder_dense_2(x)
        z_mean = self.z_mean_layer(x)
        z_log_var = self.z_log_var_layer(x)
        z = layers.Lambda(self.sampling, output_shape=(self.latent_dim,))([z_mean, z_log_var])

        h_decoded = self.decoder_dense_1(z)
        x_decoded_mean = self.decoder_output_layer(h_decoded)

        reconstruction_loss = tf.reduce_sum(tf.keras.losses.binary_crossentropy(features, x_decoded_mean), axis=-1)
        reconstruction_loss = tf.reduce_mean(reconstruction_loss)

        kl_loss = -0.5 * tf.reduce_sum(z_log_var - tf.square(z_mean) - tf.exp(z_log_var) + 1, axis=-1)
        kl_loss = tf.reduce_mean(kl_loss)


        self.add_loss(reconstruction_loss)
        self.add_loss(kl_loss)

        return x_decoded_mean

unique_genres = genre_column.unique()
genre_to_int = {genre: i for i, genre in enumerate(unique_genres)}
genre_int_encoded = genre_column.map(genre_to_int).values
num_genres = len(unique_genres)
genre_one_hot = to_categorical(genre_int_encoded, num_classes=num_genres)

original_dim = combined_features_scaled.shape[1]

cvae_model = CVAE(original_dim=original_dim, latent_dim=latent_dim, num_genres=num_genres)

cvae_model.compile(optimizer='rmsprop')

cvae_model.fit(x=[combined_features_scaled, genre_one_hot],
               y=combined_features_scaled,
               epochs=50,
               batch_size=64)

In [ ]:
input_features = layers.Input(shape=(combined_features_scaled.shape[1],), name='encoder_input_features')
input_genre_one_hot = layers.Input(shape=(num_genres,), name='encoder_input_genre_one_hot')

x = cvae_model.encoder_input_concat([input_features, input_genre_one_hot])
x = cvae_model.encoder_dense_1(x)
x = cvae_model.encoder_dense_2(x)
z_mean_output = cvae_model.z_mean_layer(x)

encoder_model = models.Model(inputs=[input_features, input_genre_one_hot], outputs=z_mean_output)

latent_features = encoder_model.predict([combined_features_scaled, genre_one_hot])

latent_features_df = pd.DataFrame(latent_features, columns=[f'latent_dim_{i+1}' for i in range(latent_dim)])

print(latent_features_df.head())

Multi Modal Clustering

In [ ]:
multi_modal_features = pd.concat([combined_features, latent_features_df], axis=1)
print(f"Shape of multi-modal features: {multi_modal_features.shape}")

Silhouette Score K means

In [ ]:
kmeans = KMeans(n_clusters=3, random_state=42)
kmeans_labels = kmeans.fit_predict(multi_modal_features)

sil_score_kmeans = silhouette_score(multi_modal_features, kmeans_labels)
print(f'Silhouette Score (K-Means): {sil_score_kmeans}')

Agglomerative Clustering

In [ ]:
agg_clust = AgglomerativeClustering(n_clusters=3)
agg_labels = agg_clust.fit_predict(multi_modal_features)

agg_sil_score = silhouette_score(multi_modal_features, agg_labels)
print(f'Agglomerative Clustering Silhouette Score: {agg_sil_score}')

DBSCAN Clustering

In [ ]:
dbscan = DBSCAN(eps=0.2, min_samples=3)
db_labels = dbscan.fit_predict(multi_modal_features)

print(f"Number of clusters found: {len(set(db_labels))}")

if len(set(db_labels)) > 1:
    db_sil_score = silhouette_score(multi_modal_features, db_labels)
    print(f'DBSCAN Silhouette Score: {db_sil_score}')
else:
    print("Silhouette Score cannot be calculated because there is only one cluster.")

Davies Bouldin Index

In [ ]:
# from sklearn.metrics import davies_bouldin_score
db_index_kmeans = davies_bouldin_score(multi_modal_features, kmeans_labels)
print(f'Davies-Bouldin Index (K-Means): {db_index_kmeans}')

Adjusted Rand Index

In [ ]:
# from sklearn.metrics import adjusted_rand_score
true_labels = data['genre']
ari_score = adjusted_rand_score(true_labels, kmeans_labels)
print(f'Adjusted Rand Index (ARI): {ari_score}')

Visualization

In [ ]:
# from sklearn.manifold import TSNE
tsne = TSNE(n_components=2, random_state=42)
latent_2d_tsne = tsne.fit_transform(multi_modal_features)

plt.figure(figsize=(8, 6))
plt.scatter(latent_2d_tsne[:, 0], latent_2d_tsne[:, 1], c=kmeans_labels, cmap='viridis', marker='o')
plt.title('Multi-modal Clustering of Latent Features')
plt.xlabel('Dimension 1')
plt.ylabel('Dimension 2')
plt.show()

Cluster Purity

In [ ]:
def calculate_cluster_purity(true_labels, predicted_labels):

    cm = confusion_matrix(true_labels, predicted_labels)
    cluster_purity = np.sum(np.max(cm, axis=1)) / np.sum(cm)
    return cluster_purity
true_labels_encoded = genre_int_encoded

cluster_purity_kmeans = calculate_cluster_purity(true_labels_encoded, kmeans_labels)
print(f'Cluster Purity (K-Means): {cluster_purity_kmeans}')

cluster_purity_agg = calculate_cluster_purity(true_labels_encoded, agg_labels)
print(f'Cluster Purity (Agglomerative): {cluster_purity_agg}')

cluster_purity_dbscan = calculate_cluster_purity(true_labels_encoded, db_labels)
print(f'Cluster Purity (DBSCAN): {cluster_purity_dbscan}')

Latent Space Plot

In [ ]:
tsne = TSNE(n_components=2, random_state=42)
latent_2d_tsne = tsne.fit_transform(latent_features)

plt.figure(figsize=(8, 6))
plt.scatter(latent_2d_tsne[:, 0], latent_2d_tsne[:, 1], c=kmeans_labels, cmap='viridis', marker='o')
plt.title('Latent Space of Conditional VAE (t-SNE)')
plt.xlabel('Latent Dimension 1')
plt.ylabel('Latent Dimension 2')
plt.colorbar(label='Cluster')
plt.show()

Cluster Distribution over language-genre

In [ ]:
latent_features_df['genre'] = data['genre']
latent_features_df['cluster'] = kmeans_labels

plt.figure(figsize=(10, 6))
plt.scatter(latent_features_df['latent_dim_1'], latent_features_df['latent_dim_2'], c=latent_features_df['cluster'], cmap='viridis', marker='o')
plt.title('Cluster Distribution over Genres')
plt.xlabel('Latent Dimension 1')
plt.ylabel('Latent Dimension 2')
plt.colorbar(label='Cluster')
plt.show()

for genre in latent_features_df['genre'].unique():
    subset = latent_features_df[latent_features_df['genre'] == genre]
    plt.scatter(subset['latent_dim_1'], subset['latent_dim_2'], label=genre)

plt.legend(loc='best')
plt.title('Cluster Distribution over Genres')
plt.xlabel('Latent Dimension 1')
plt.ylabel('Latent Dimension 2')
plt.show()

Reconstruction

In [ ]:
num_samples = 5
latent_samples = latent_features[:num_samples]

latent_input = tf.keras.layers.Input(shape=(latent_dim,), name='latent_space_input')

decoder_output = cvae_model.decoder_output_layer(cvae_model.decoder_dense_1(latent_input))

decoder_model = tf.keras.models.Model(latent_input, decoder_output)
reconstructed_samples = decoder_model.predict(latent_samples)

for i in range(num_samples):
    plt.figure(figsize=(12, 4))
    plt.subplot(1, 2, 1)
    plt.plot(combined_features_scaled[i], label="Original")
    plt.title(f"Original Sample {i+1}")

    plt.subplot(1, 2, 2)
    plt.plot(reconstructed_samples[i], label="Reconstructed")
    plt.title(f"Reconstructed Sample {i+1}")

    plt.tight_layout()
    plt.show()

pca+k means scores


In [ ]:
pca = PCA(n_components=2, random_state=42)
latent_2d_pca = pca.fit_transform(combined_features_scaled)

kmeans_pca = KMeans(n_clusters=3, random_state=42)
kmeans_labels_pca = kmeans_pca.fit_predict(latent_2d_pca)

sil_score_pca = silhouette_score(latent_2d_pca, kmeans_labels_pca)
db_index_pca = davies_bouldin_score(latent_2d_pca, kmeans_labels_pca)
ari_score_pca = adjusted_rand_score(true_labels, kmeans_labels_pca) if 'true_labels' in locals() else None

print(f"PCA + K-Means Silhouette Score: {sil_score_pca}")
print(f"PCA + K-Means Davies-Bouldin Index: {db_index_pca}")
print(f"PCA + K-Means ARI Score: {ari_score_pca if ari_score_pca is not None else 'N/A'}")

Autoencoder+k means scores


In [ ]:
input_dim = combined_features_scaled.shape[1]
encoding_dim = 2

input_layer = layers.Input(shape=(input_dim,))
encoded = layers.Dense(64, activation='relu')(input_layer)
encoded = layers.Dense(32, activation='relu')(encoded)
latent = layers.Dense(encoding_dim, activation='relu')(encoded)

decoded = layers.Dense(32, activation='relu')(latent)
decoded = layers.Dense(64, activation='relu')(decoded)
decoded = layers.Dense(input_dim, activation='sigmoid')(decoded)

autoencoder = models.Model(input_layer, decoded)
encoder = models.Model(input_layer, latent)

autoencoder.compile(optimizer='adam', loss='mse')
autoencoder.fit(combined_features_scaled, combined_features_scaled, epochs=50, batch_size=64)

latent_features_autoencoder = encoder.predict(combined_features_scaled)

kmeans_autoencoder = KMeans(n_clusters=3, random_state=42)
kmeans_labels_autoencoder = kmeans_autoencoder.fit_predict(latent_features_autoencoder)

sil_score_autoencoder = silhouette_score(latent_features_autoencoder, kmeans_labels_autoencoder)
db_index_autoencoder = davies_bouldin_score(latent_features_autoencoder, kmeans_labels_autoencoder)
ari_score_autoencoder = adjusted_rand_score(true_labels, kmeans_labels_autoencoder) if 'true_labels' in locals() else None

print(f"Autoencoder + K-Means Silhouette Score: {sil_score_autoencoder}")
print(f"Autoencoder + K-Means Davies-Bouldin Index: {db_index_autoencoder}")
print(f"Autoencoder + K-Means ARI Score: {ari_score_autoencoder if ari_score_autoencoder is not None else 'N/A'}")

MFCC + K-Means scores

In [ ]:
kmeans_mfcc = KMeans(n_clusters=3, random_state=42)
kmeans_labels_mfcc = kmeans_mfcc.fit_predict(combined_features_scaled)

sil_score_mfcc = silhouette_score(combined_features_scaled, kmeans_labels_mfcc)
db_index_mfcc = davies_bouldin_score(combined_features_scaled, kmeans_labels_mfcc)
ari_score_mfcc = adjusted_rand_score(true_labels, kmeans_labels_mfcc) if 'true_labels' in locals() else None

print(f"MFCC + K-Means Silhouette Score: {sil_score_mfcc}")
print(f"MFCC + K-Means Davies-Bouldin Index: {db_index_mfcc}")
print(f"MFCC + K-Means ARI Score: {ari_score_mfcc if ari_score_mfcc is not None else 'N/A'}")

VAE + K-Means scores

In [ ]:
kmeans_vae = KMeans(n_clusters=3, random_state=42)
kmeans_labels_vae = kmeans_vae.fit_predict(latent_features)

sil_score_vae = silhouette_score(latent_features, kmeans_labels_vae)
db_index_vae = davies_bouldin_score(latent_features, kmeans_labels_vae)
ari_score_vae = adjusted_rand_score(true_labels, kmeans_labels_vae) if 'true_labels' in locals() else None

print(f"VAE + K-Means Silhouette Score: {sil_score_vae}")
print(f"VAE + K-Means Davies-Bouldin Index: {db_index_vae}")
print(f"VAE + K-Means ARI Score: {ari_score_vae if ari_score_vae is not None else 'N/A'}")

In [ ]:
results = {
    'Method': ['PCA + K-Means', 'Autoencoder + K-Means', 'MFCC + K-Means', 'VAE-based Clustering'],
    'Silhouette Score': [sil_score_pca, sil_score_autoencoder, sil_score_mfcc, sil_score_vae],
    'Davies-Bouldin Index': [db_index_pca, db_index_autoencoder, db_index_mfcc, db_index_vae],
    'Adjusted Rand Index': [ari_score_pca if ari_score_pca is not None else 'N/A',
                            ari_score_autoencoder if ari_score_autoencoder is not None else 'N/A',
                            ari_score_mfcc if ari_score_mfcc is not None else 'N/A',
                            ari_score_vae if ari_score_vae is not None else 'N/A']
}

comparison_df = pd.DataFrame(results)

print(comparison_df)

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(18, 6))

ax[0].bar(comparison_df['Method'], comparison_df['Silhouette Score'], color='lightblue')
ax[0].set_title('Silhouette Score Comparison')
ax[0].set_ylabel('Silhouette Score')

ax[1].bar(comparison_df['Method'], comparison_df['Davies-Bouldin Index'], color='lightgreen')
ax[1].set_title('Davies-Bouldin Index Comparison')
ax[1].set_ylabel('Davies-Bouldin Index')

ax[2].bar(comparison_df['Method'], comparison_df['Adjusted Rand Index'], color='lightcoral')
ax[2].set_title('Adjusted Rand Index Comparison')
ax[2].set_ylabel('Adjusted Rand Index')

plt.tight_layout()
plt.show()